# 06 — Transduktif -> Induktif Fayda Olcumu (Inductive Utility)

**Cold-Start Sosyal-Grafik Sizintisi — Durust Fayda Ekseni**

Makale recommender faydasini (Pareto egrilerinin dayandigi link-prediction AUC) **transduktif** olcer: model tum grafikle egitilir ve gordugu kenarlar uzerinde degerlendirilir. Bu, genellemeyi degil ezberlemeyi olcer ve faydayi sisirir. Bu defter ayni modeli **induktif** olarak degerlendirir:

- Kenarlarin bir kismi (test_frac) egitimden tamamen tutulur (held-out): ne mesaj-gecisinde ne BPR denetiminde kullanilir.
- **Transduktif AUC**: pozitifler egitim kenarlarindan (makalenin raporladigi).
- **Induktif AUC**: pozitifler held-out kenarlardan (model hic gormedi).
- **Recall@20 / NDCG@20 (induktif)**: her dugum icin bilinen (egitim) komsulari sirlamadan cikar, held-out komsulari top-20 ile ne kadar geri getirdigini olcer = recommender'in *gercekte* yaptigi is.
- Ayni train-only modelde **sizinti** (cold-start precision/recall) da olculur; boylece mahremiyet ve durust fayda **tek model** uzerinde eslesir.

**Sadakat:** uretici + LightGCN/FeatureGNN + BPR 02b ile birebir; tek fark egitimin yalniz train kenarlariyla yapilmasi.

## 0. Kurulum

In [ ]:
!pip install -q networkx numpy pandas scipy torch matplotlib

In [ ]:
import os, json, time, math
import numpy as np, pandas as pd, networkx as nx
import torch, torch.nn as nn
def _in_colab():
    try:
        import google.colab; return True
    except Exception: return False
if _in_colab():
    from google.colab import drive; drive.mount('/content/drive'); ROOT='/content/drive/MyDrive/cold_start_leakage'
else: ROOT=os.path.abspath('./cold_start_leakage')
OUT_DIR=os.path.join(ROOT,'inductive'); os.makedirs(OUT_DIR,exist_ok=True)
print('Cikti:',OUT_DIR,'| torch',torch.__version__)

## 1. Ayarlar

In [ ]:
MODE="quick"          # "quick" | "full"
BASELINE=dict(p_in=0.06,p_out=0.006,recip=0.49,homo=0.70,priv=0.30)
K=20
TEST_FRAC=0.15        # held-out kenar orani
if MODE=="quick":
    SEEDS=[0,1,2]; GRAPHS=["sbm"]; MODELS=["lightgcn","featuregnn"]; EPOCHS=30
else:
    SEEDS=list(range(10)); GRAPHS=["sbm","lfr"]; MODELS=["lightgcn","featuregnn"]; EPOCHS=45
N_NODES=700
print(f"MODE={MODE} | {len(SEEDS)} seed x {len(GRAPHS)} grafik x {len(MODELS)} model")

## 2. Uretici + modeller (02b ile birebir)

In [ ]:
class DS:
    def __init__(s,G,priv,comm,nl,X): s.G,s.private,s.community,s.node_list,s.features=G,priv,comm,nl,X
    def neigh(s,t): return set(s.G.predecessors(t))|set(s.G.successors(t))
def _orient(Gu,r,rng):
    D=nx.DiGraph(); D.add_nodes_from(Gu.nodes())
    for u,v in Gu.edges():
        if u==v: continue
        if rng.random()<r: D.add_edge(u,v); D.add_edge(v,u)
        else: (D.add_edge(u,v) if rng.random()<0.5 else D.add_edge(v,u))
    return D
def _feat(n,nc,comm,homo,fd,rng):
    mu=rng.normal(size=(nc,fd)); X=np.zeros((n,fd))
    for i in range(n): X[i]=homo*mu[comm[i]]+(1-homo)*rng.normal(size=fd)
    return ((X-X.mean(0))/(X.std(0)+1e-9)).astype(np.float32)
def _lab(n,priv,rng):
    pr={i:False for i in range(n)}
    for i in rng.choice(n,int(round(priv*n)),replace=False): pr[i]=True
    return pr
def make_sbm(n=700,nc=7,p_in=0.06,p_out=0.006,recip=0.49,homo=0.70,priv=0.30,fd=16,seed=0):
    rng=np.random.default_rng(seed); sizes=[n//nc]*nc
    for i in range(n-sum(sizes)): sizes[i]+=1
    P=np.full((nc,nc),p_out); np.fill_diagonal(P,p_in)
    Gu=nx.stochastic_block_model(sizes,P,seed=int(seed)); comm={i:Gu.nodes[i]['block'] for i in Gu.nodes()}
    return DS(_orient(Gu,recip,rng),_lab(n,priv,rng),comm,list(range(n)),_feat(n,nc,comm,homo,fd,rng))
def make_lfr(n=700,recip=0.49,homo=0.70,priv=0.30,fd=16,seed=0,mu=0.1,**_):
    rng=np.random.default_rng(seed); Gu=None
    for off in range(8):
        try:
            Gu=nx.LFR_benchmark_graph(n=n,tau1=3,tau2=1.5,mu=mu,average_degree=10,min_community=30,seed=int(seed)+off,max_iters=500); break
        except Exception: continue
    if Gu is None: return None
    Gu=nx.Graph(Gu); Gu.remove_edges_from(nx.selfloop_edges(Gu)); Gu=nx.convert_node_labels_to_integers(Gu)
    raw={i:frozenset(Gu.nodes[i]["community"]) for i in Gu.nodes()}; uq={c:j for j,c in enumerate(set(raw.values()))}
    comm={i:uq[raw[i]] for i in Gu.nodes()}
    return DS(_orient(Gu,recip,rng),_lab(n,priv,rng),comm,list(range(n)),_feat(n,len(uq),comm,homo,fd,rng))
class LightGCN(nn.Module):
    def __init__(s,n,dim=64,layers=3):
        super().__init__(); s.emb=nn.Embedding(n,dim); nn.init.normal_(s.emb.weight,std=0.1); s.layers=layers
    def forward(s,Ah):
        e=s.emb.weight; outs=[e]
        for _ in range(s.layers): e=Ah@e; outs.append(e)
        return torch.stack(outs,0).mean(0)
class FeatureGNN(nn.Module):
    def __init__(s,n,fd,dim=64,layers=2):
        super().__init__(); s.id=nn.Embedding(n,dim); nn.init.normal_(s.id.weight,std=0.1)
        s.fp=nn.Linear(fd,dim); s.W=nn.ModuleList([nn.Linear(2*dim,dim) for _ in range(layers)]); s.act=nn.ReLU(); s.layers=layers
    def forward(s,Ah,Xf):
        h=s.id.weight+s.fp(Xf)
        for l in range(s.layers): h=s.act(s.W[l](torch.cat([h,Ah@h],1)))
        return h
def norm_adj_edges(edges,n):
    A=torch.zeros((n,n))
    for u,v in edges: A[u,v]=1.0; A[v,u]=1.0
    A+=torch.eye(n); deg=A.sum(1); di=torch.pow(deg,-0.5); di[torch.isinf(di)]=0
    return torch.diag(di)@A@torch.diag(di)
def train(model,Ah,pos,n,Xf=None,epochs=45,bs=1024,lr=0.01):
    opt=torch.optim.Adam(model.parameters(),lr=lr,weight_decay=1e-5); pos=torch.tensor(pos,dtype=torch.long)
    for ep in range(epochs):
        perm=torch.randperm(pos.size(0))
        for i in range(0,pos.size(0),bs):
            idx=perm[i:i+bs]; u,p=pos[idx,0],pos[idx,1]; neg=torch.randint(0,n,(u.size(0),))
            E=model(Ah) if Xf is None else model(Ah,Xf)
            su=(E[u]*E[p]).sum(1); sn=(E[u]*E[neg]).sum(1)
            loss=-torch.log(torch.sigmoid(su-sn)+1e-9).mean(); opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad(): return (model(Ah) if Xf is None else model(Ah,Xf)).detach().numpy()

## 3. Olcumler: AUC (trans/ind), Recall/NDCG@20 (ind), sizinti

In [ ]:
def auc(En,pos_edges,full_set,nodes,rng,n_samples=2000):
    pe=[pos_edges[i] for i in rng.choice(len(pos_edges),size=min(n_samples,len(pos_edges)),replace=False)]
    sc=lambda a,b: float(En[a]@En[b]); ps=np.array([sc(u,v) for u,v in pe]); ns=[]
    while len(ns)<len(ps):
        a,b=rng.choice(nodes,2,replace=False)
        if frozenset((int(a),int(b))) not in full_set: ns.append(sc(int(a),int(b)))
    ns=np.array(ns); return float((ps[:,None]>ns[None,:]).mean())
def ndcg_recall_ind(En,train_adj,test_adj,nodes,k=20):
    rec,nd=[],[]
    for u in nodes:
        tn=test_adj.get(u,set())
        if not tn: continue
        s=En@En[u]; s[u]=-1e9
        for x in train_adj.get(u,set()): s[x]=-1e9
        top=np.argpartition(-s,k)[:k]; top=top[np.argsort(-s[top])]
        hit=[1 if int(x) in tn else 0 for x in top]; rec.append(sum(hit)/len(tn))
        dcg=sum(h/math.log2(i+2) for i,h in enumerate(hit)); idcg=sum(1/math.log2(i+2) for i in range(min(k,len(tn))))
        nd.append(dcg/idcg if idcg>0 else 0.0)
    return float(np.mean(rec)),float(np.mean(nd)),len(rec)
def leakage(En,ds,k=20,seed=0):
    nodes=ds.node_list; tg=[t for t in nodes if ds.private[t] and ds.G.degree(t)>=4]; cp,cr=[],[]
    for t in tg:
        Nt=ds.neigh(t)
        if not Nt: continue
        s=En@En[t]; s[t]=-1e9; top=set(np.argpartition(-s,k)[:k].tolist())
        cp.append(len(top&Nt)/k); cr.append(len(top&Nt)/len(Nt))
    return float(np.mean(cp)),float(np.mean(cr))
def run_one(graph,model_name,seed,test_frac=TEST_FRAC,epochs=EPOCHS):
    torch.manual_seed(seed); np.random.seed(seed)
    ds=(make_sbm(n=N_NODES,seed=seed,**BASELINE) if graph=="sbm" else make_lfr(n=N_NODES,seed=seed,homo=BASELINE["homo"],priv=BASELINE["priv"],recip=BASELINE["recip"]))
    if ds is None: return None
    n=len(ds.node_list); nodes=ds.node_list; und=ds.G.to_undirected(); edges=[(u,v) for u,v in und.edges()]
    rng=np.random.default_rng(seed+1); perm=rng.permutation(len(edges)); nt=int(test_frac*len(edges))
    test_e=[edges[i] for i in perm[:nt]]; train_e=[edges[i] for i in perm[nt:]]
    full_set=set(frozenset((u,v)) for u,v in edges); Ah=norm_adj_edges(train_e,n)
    if model_name=="lightgcn": E=train(LightGCN(n),Ah,train_e,n,epochs=epochs)
    else:
        Xf=torch.tensor(ds.features); E=train(FeatureGNN(n,ds.features.shape[1]),Ah,train_e,n,Xf=Xf,epochs=epochs)
    En=E/(np.linalg.norm(E,axis=1,keepdims=True)+1e-9); rng2=np.random.default_rng(seed+2)
    train_adj={}; test_adj={}
    for u,v in train_e: train_adj.setdefault(u,set()).add(v); train_adj.setdefault(v,set()).add(u)
    for u,v in test_e: test_adj.setdefault(u,set()).add(v); test_adj.setdefault(v,set()).add(u)
    rec,nd,nev=ndcg_recall_ind(En,train_adj,test_adj,nodes); lp,lr=leakage(En,ds,seed=seed)
    return dict(graph=graph,model=model_name,seed=seed,
                auc_trans=auc(En,train_e,full_set,nodes,rng2),auc_ind=auc(En,test_e,full_set,nodes,rng2),
                recall20_ind=rec,ndcg20_ind=nd,n_eval=nev,leak_prec=lp,leak_recall=lr)

## 4. ETA + cok-seed kosu

In [ ]:
_t=time.time(); _=run_one(GRAPHS[0],MODELS[0],SEEDS[0]); per=time.time()-_t
total=len(SEEDS)*len(GRAPHS)*len(MODELS)
print(f"~{per:.1f}s/kosu | {total} kosu | tahmini ~{per*total/60:.1f} dk")

In [ ]:
rows=[]; t0=time.time(); done=0
for gt in GRAPHS:
    for model in MODELS:
        for seed in SEEDS:
            r=run_one(gt,model,seed); done+=1
            if r is None: print(f"[atlandi] {gt} {model} seed={seed}"); continue
            r["auc_gap"]=r["auc_trans"]-r["auc_ind"]; rows.append(r)
            pd.DataFrame(rows).to_csv(os.path.join(OUT_DIR,"inductive_long.csv"),index=False)
            print(f"[{done}/{total}] {gt} {model:10s} seed={seed} AUCtrans={r['auc_trans']:.3f} "
                  f"AUCind={r['auc_ind']:.3f} gap={r['auc_gap']:+.3f} R@20ind={r['recall20_ind']:.3f} "
                  f"NDCG={r['ndcg20_ind']:.3f} | leak_prec={r['leak_prec']:.3f} leak_rec={r['leak_recall']:.3f}")
df=pd.DataFrame(rows); print(f"\nBitti: {len(df)} satir, {time.time()-t0:.1f}s")

## 5. Toplulastir + Claude'a yapistirilacak ozet

In [ ]:
def ci95(x):
    x=np.asarray(x,float); n=len(x); return 1.96*x.std(ddof=1)/math.sqrt(n) if n>1 else 0.0
df=pd.read_csv(os.path.join(OUT_DIR,"inductive_long.csv"))
def A(col,g,m):
    v=df[(df.graph==g)&(df.model==m)][col].values; return (float(np.mean(v)),ci95(v)) if len(v) else (float('nan'),0.0)
graphs=sorted(df.graph.unique()); models=sorted(df.model.unique())
L=["### 1) Fayda: transduktif vs induktif (mu+-CI; seed'ler arasi)","",
   "| grafik | model | AUC_trans | AUC_ind | gap (sisme) | Recall@20_ind | NDCG@20_ind |",
   "|---|---|---|---|---|---|---|"]
for g in graphs:
    for m in models:
        at=A("auc_trans",g,m); ai=A("auc_ind",g,m); gp=A("auc_gap",g,m); rc=A("recall20_ind",g,m); nd=A("ndcg20_ind",g,m)
        L.append(f"| {g} | {m} | {at[0]:.3f}+-{at[1]:.3f} | {ai[0]:.3f}+-{ai[1]:.3f} | {gp[0]:+.3f}+-{gp[1]:.3f} | "
                 f"{rc[0]:.3f}+-{rc[1]:.3f} | {nd[0]:.3f}+-{nd[1]:.3f} |")
L+=["","### 2) Ayni train-only modelde mahremiyet (sizinti devam ediyor mu?)","",
    "| grafik | model | leak_prec@20 | leak_recall@20 |","|---|---|---|---|"]
for g in graphs:
    for m in models:
        lp=A("leak_prec",g,m); lr=A("leak_recall",g,m)
        L.append(f"| {g} | {m} | {lp[0]:.3f}+-{lp[1]:.3f} | {lr[0]:.3f}+-{lr[1]:.3f} |")
meta=[f"# Cold-Start — Inductive Utility ozeti","",
      f"- MODE={MODE} | K={K} | test_frac={TEST_FRAC} | seeds={len(SEEDS)} | graphs={graphs} | epochs={EPOCHS}",""]
share="\n".join(meta+L)+"\n"
open(os.path.join(OUT_DIR,"SHARE_WITH_CLAUDE.md"),"w").write(share)
print("="*70); print(share); print("="*70); print("Iki ===== arasini kopyalayip Claude'a gonder.")

## 6. (Opsiyonel) Figur — sisme + durust tradeoff

In [ ]:
import matplotlib.pyplot as plt
df=pd.read_csv(os.path.join(OUT_DIR,"inductive_long.csv"))
graphs=sorted(df.graph.unique()); models=sorted(df.model.unique())
fig,ax=plt.subplots(1,2,figsize=(11,4))
labels=[f"{g}\n{m}" for g in graphs for m in models]; x=np.arange(len(labels)); w=0.35
tr=[df[(df.graph==g)&(df.model==m)]["auc_trans"].mean() for g in graphs for m in models]
ind=[df[(df.graph==g)&(df.model==m)]["auc_ind"].mean() for g in graphs for m in models]
ax[0].bar(x-w/2,tr,w,label="transduktif"); ax[0].bar(x+w/2,ind,w,label="induktif")
ax[0].axhline(0.5,ls=":",c="gray",label="sans (0.5)"); ax[0].set_xticks(x); ax[0].set_xticklabels(labels,fontsize=8)
ax[0].set_ylabel("link-prediction AUC"); ax[0].set_title("Fayda sismesi: trans vs ind"); ax[0].legend()
for g in graphs:
    for m in models:
        d=df[(df.graph==g)&(df.model==m)]
        ax[1].scatter(d["recall20_ind"].mean(),d["leak_recall"].mean(),s=70)
        ax[1].annotate(f"{g}/{m}",(d["recall20_ind"].mean(),d["leak_recall"].mean()),fontsize=8,xytext=(4,4),textcoords="offset points")
ax[1].set_xlabel("durust fayda: Recall@20 (induktif)"); ax[1].set_ylabel("mahremiyet kaybi: sizinti recall@20")
ax[1].set_title("Durust tradeoff (sag-alt = kotu)"); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR,"inductive_figs.png"),dpi=130); plt.show()
print("[kaydedildi]",os.path.join(OUT_DIR,"inductive_figs.png"))